In [5]:
import base64
import gzip
import requests

url = "https://api.starglass.cfa.harvard.edu/public/dasch/dr7/cutout"

payload = {
    "plate_id": "b02312",        # example
    "solution_number": 0,        # example
    "center_ra_deg": 83.8095,   # your target RA in degrees
    "center_dec_deg": -5.3939     # your target Dec in degrees
}

r = requests.post(url, json=payload, headers={"Accept": "application/json"})
r.raise_for_status()

# The DASCH client code decodes the API response this way:
fits_bytes = gzip.decompress(base64.b64decode(r.json()))

with open("cutout.fits", "wb") as f:
    f.write(fits_bytes)

In [4]:
import base64
import gzip
import requests

BASE = "https://api.starglass.cfa.harvard.edu/public/dasch/dr7"

# Orion Nebula (M42), J2000/ICRS-ish decimal degrees
ra_deg = 83.8095
dec_deg = -5.3939

# Step 1: find exposures overlapping the target position
q = requests.post(
    f"{BASE}/queryexps",
    json={
        "center_ra_deg": ra_deg,
        "center_dec_deg": dec_deg,
    },
    headers={"Accept": "application/json"},
)
q.raise_for_status()

rows = q.json()

# DASCH tabular APIs usually return a list of CSV-like rows:
# first row = header, remaining rows = data
header = rows[0]
data = rows[1:]

# Turn rows into dicts
records = [dict(zip(header, row)) for row in data]

# Step 2: look for plate b02312 with a valid solution number
matches = [
    r for r in records
    if (r.get("series", "") + str(r.get("platenum", "")).zfill(5)).lower() == "b02312"
    and str(r.get("solnum", "")).strip() not in ("", "None", "-1")
]

print(f"Found {len(matches)} matching exposure(s) for b02312")

for m in matches[:10]:
    print(
        "plate_id=", (m.get("series", "") + str(m.get("platenum", "")).zfill(5)).lower(),
        "solnum=", m.get("solnum"),
        "expnum=", m.get("expnum"),
        "ra=", m.get("ctr_ra"),
        "dec=", m.get("ctr_dec"),
    )

if not matches:
    raise RuntimeError("No overlapping solved exposure found for b02312 at Orion Nebula coordinates")

solnum = int(matches[0]["solnum"])

# Step 3: request the cutout using the verified solution number
payload = {
    "plate_id": "b02312",
    "solution_number": solnum,
    "center_ra_deg": ra_deg,
    "center_dec_deg": dec_deg,
}

r = requests.post(
    f"{BASE}/cutout",
    json=payload,
    headers={"Accept": "application/json"},
)

print("status:", r.status_code)
print("content-type:", r.headers.get("content-type"))
print("text preview:", r.text[:300])

r.raise_for_status()

fits_bytes = gzip.decompress(base64.b64decode(r.json()))
with open("cutout.fits", "wb") as f:
    f.write(fits_bytes)

print("Wrote cutout.fits")

HTTPError: 502 Server Error: Bad Gateway for url: https://api.starglass.cfa.harvard.edu/public/dasch/dr7/queryexps